# Конспект 16.1 — Семантическая сегментация: интерактивная версия

Этот ноутбук — самодостаточный конспект к домашнему заданию 16.1 по семантической сегментации. Здесь мы:

1. Реализуем метрики **IoU** и **Dice** на синтетических масках 32x32 и проверим против `torchmetrics`.
2. Реализуем **BCE Loss** в наивной (через сигмоиду) и численно стабильной форме (через `softplus`); сравним с `nn.BCEWithLogitsLoss`.
3. Реализуем **Dice Loss** и убедимся, что `dice_loss` и `dice_score` — разные сущности.
4. Реализуем **Focal Loss** и покажем его пользу при сильном дисбалансе классов.
5. Соберём упрощённые версии **SegNet** и **UNet** (3–4 уровня вместо 5) и проверим smoke-тестом.
6. Запустим короткий **демо-train** на синтетических кругах — увидим, что лосс падает.
7. Реализуем три **бонусных лосса** из научных статей: Tversky, Combo, Boundary.

Ноутбук независим от `solution.ipynb` и не требует ни датасета PH2, ни предобученных весов. Все ячейки можно запускать поодиночке.

**Запуск**: kernel `mfti`. Для воспроизводимости везде используем `torch.manual_seed(42)` / `np.random.seed(42)`.

## 0. Импорты и настройка

In [ ]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# torchmetrics — официальные реализации метрик, нужны для проверки наших
from torchmetrics import JaccardIndex

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Auto-detect device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}, device: {device}')

## 1. Метрики IoU и Dice

Семантическая сегментация — это попиксельная классификация. Качество модели оценивается тем, насколько хорошо предсказанная маска совпадает с эталонной. Две стандартные метрики:

**IoU (Intersection over Union, индекс Жаккара):**

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{TP}{TP + FP + FN}$$

**Dice Coefficient (попиксельная F1):**

$$\text{Dice} = \frac{2|A \cap B|}{|A| + |B|} = \frac{2\,TP}{2\,TP + FP + FN}$$

Связь: $\text{Dice} = \dfrac{2 \cdot \text{IoU}}{1 + \text{IoU}}$. Обе монотонны друг относительно друга — ранжируют модели одинаково, но Dice мягче (даёт более высокие значения за то же качество). Accuracy для сегментации не годится: при дисбалансе 90/10 тривиальный «всё фон» даёт accuracy = 0.9.

В коде ниже мы реализуем IoU и Dice вручную и сверим с `torchmetrics.JaccardIndex` на синтетической маске 32x32 — это два круга, частично перекрывающихся.

In [ ]:
def iou_score_simple(pred, target, eps=1e-7):
    # IoU для бинарных масок (значения 0/1)
    pred = pred.float()
    target = target.float()
    intersection = (pred * target).sum().item()
    union = pred.sum().item() + target.sum().item() - intersection
    return (intersection + eps) / (union + eps)


def dice_score_simple(pred, target, eps=1e-7):
    # Dice coefficient для бинарных масок
    pred = pred.float()
    target = target.float()
    intersection = (pred * target).sum().item()
    return (2.0 * intersection + eps) / (pred.sum().item() + target.sum().item() + eps)


# Синтетические маски: два круга, частично перекрывающихся
H = W = 32
yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing='ij')

# Эталонная маска — круг радиуса 8 в (12, 12)
gt = ((yy - 12) ** 2 + (xx - 12) ** 2 < 8 ** 2).float()
# Предсказанная маска — круг радиуса 8 в (16, 16) — смещена
pred = ((yy - 16) ** 2 + (xx - 16) ** 2 < 8 ** 2).float()

iou_my = iou_score_simple(pred, gt)
dice_my = dice_score_simple(pred, gt)

# Проверка против torchmetrics: формат (N, ...) с .int() для target
iou_tm = JaccardIndex(threshold=0.5, task='binary')(pred.unsqueeze(0), gt.unsqueeze(0).int()).item()

print(f'Наш IoU:   {iou_my:.4f}')
print(f'TM  IoU:   {iou_tm:.4f}')
print(f'Наш Dice:  {dice_my:.4f}')
print(f'Связь:     2*IoU/(1+IoU) = {2 * iou_my / (1 + iou_my):.4f}')

assert abs(iou_my - iou_tm) < 1e-4, 'Наш IoU расходится с torchmetrics'
print('ASSERT OK: наш IoU совпал с torchmetrics.JaccardIndex')

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(gt, cmap='gray');   axes[0].set_title('Ground truth')
axes[1].imshow(pred, cmap='gray'); axes[1].set_title('Prediction')
axes[2].imshow(pred + 2 * gt, cmap='viridis'); axes[2].set_title(f'Overlap: IoU={iou_my:.3f}')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

**Попробуй сам**: измени радиусы кругов или их центры. Что произойдёт с IoU при полном совпадении? При полном непересечении? Найди такую конфигурацию, при которой IoU = 0.5.

## 2. BCE Loss — наивная и стабильная реализация

Стандартный лосс для бинарной классификации (включая попиксельную) — Binary Cross-Entropy:

$$\mathcal{L}_{BCE}(y, \hat y) = -\bigl[y \log \sigma(\hat y) + (1 - y)\log(1 - \sigma(\hat y))\bigr] \quad [1]$$

где $\hat y$ — **логит** (выход сети до сигмоиды), $\sigma(\hat y) = 1/(1 + e^{-\hat y})$.

**Проблема формулы [1]**: при больших $|\hat y|$ значение $\sigma(\hat y)$ упирается в 0 или 1, и `log(0) = -inf` — численная катастрофа.

**Стабильная формула** получается подстановкой логарифма сигмоиды:

$$\mathcal{L}_{BCE} = \hat y - y\hat y + \log(1 + e^{-\hat y}) \quad [2]$$

Дополнительно стабилизируем через тождество $\log(1 + e^{-\hat y}) = \max(0, -\hat y) + \log(1 + e^{-|\hat y|})$ — именно так делает `torch.nn.BCEWithLogitsLoss`. **Правило**: всегда используй `BCEWithLogitsLoss` вместо `BCELoss(sigmoid(x), y)`.

Ниже реализуем оба варианта вручную и проверим, что они совпадают с эталоном PyTorch.

In [ ]:
def bce_naive(logits, labels, eps=1e-12):
    # Наивная BCE по формуле [1] через сигмоиду — нестабильна при больших |logits|
    p = torch.sigmoid(logits)
    return -(labels * torch.log(p + eps) + (1 - labels) * torch.log(1 - p + eps)).sum()


def softplus_stable(x):
    # log(1 + exp(x)) с защитой от переполнения: max(0,x) + log(1+exp(-|x|))
    return torch.clamp(x, min=0) + torch.log1p(torch.exp(-torch.abs(x)))


def bce_stable(logits, labels):
    # Стабильная BCE по формуле [2]
    return (logits - labels * logits + softplus_stable(-logits)).sum()


# Тест 1: умеренные значения — оба варианта должны совпадать с BCEWithLogitsLoss
torch.manual_seed(SEED)
logits = torch.randn(2, 1, 8, 8)
labels = (torch.rand(2, 1, 8, 8) > 0.5).float()

v_naive  = bce_naive(logits, labels).item()
v_stable = bce_stable(logits, labels).item()
v_torch  = nn.BCEWithLogitsLoss(reduction='sum')(logits, labels).item()

print(f'naive BCE         : {v_naive:.4f}')
print(f'stable BCE        : {v_stable:.4f}')
print(f'BCEWithLogitsLoss : {v_torch:.4f}')

assert abs(v_naive  - v_torch) < 1e-3
assert abs(v_stable - v_torch) < 1e-3
print('ASSERT OK: обе реализации сошлись с эталоном PyTorch.')

In [ ]:
# Тест 2: экстремальные логиты — наивная форма ломается, стабильная — нет
extreme_logits = torch.tensor([[100.0, -100.0, 50.0, -50.0]])
extreme_labels = torch.tensor([[1.0, 0.0, 1.0, 0.0]])

print('Экстремальные логиты:', extreme_logits.tolist())
print(f'naive BCE         : {bce_naive(extreme_logits, extreme_labels).item():.4f}  (может быть inf/NaN)')
print(f'stable BCE        : {bce_stable(extreme_logits, extreme_labels).item():.4f}')
print(f'BCEWithLogitsLoss : {nn.BCEWithLogitsLoss(reduction="sum")(extreme_logits, extreme_labels).item():.4f}')

**Наблюдение**: при $\hat y = 100$ и $y = 1$ ожидаемое значение лосса близко к нулю (модель уверенно правильно предсказала). Стабильная форма даёт корректный ответ, наивная — может сломаться из-за `log(1 - 1.0) = -inf` (но в современных torch автоматически использует safe-log).

**Попробуй сам**: подставь логиты `[1000, -1000]` — увидишь, что `naive` начинает терять точность из-за float32.

## 3. Dice Loss

Dice Loss — прямой способ оптимизировать качество маски, использует **soft Dice** (без threshold) для дифференцируемости:

$$\mathcal{L}_{Dice} = 1 - \frac{2 \cdot \sum \sigma(\hat y) \cdot y + \varepsilon}{\sum \sigma(\hat y) + \sum y + \varepsilon}$$

`eps` в числителе и знаменателе нужен для двух целей:

1. избежать деления на 0 при пустых масках,
2. дать ненулевой градиент даже когда `pred * target = 0`.

**Важно**: `dice_loss` (для оптимизации, использует `sigmoid(logits)`) и `dice_score` (для метрики, использует threshold по 0.5) — **разные функции**. Нельзя путать.

In [ ]:
def dice_loss(logits, labels, eps=1.0):
    # Soft Dice Loss: 1 - 2*sum(sigmoid*labels) / (sum(sigmoid)+sum(labels))
    probs = torch.sigmoid(logits)
    intersection = (probs * labels).sum()
    denom = probs.sum() + labels.sum()
    return 1.0 - (2.0 * intersection + eps) / (denom + eps)


def dice_score_metric(logits, labels, threshold=0.5, eps=1e-7):
    # Hard Dice score (метрика): применяет threshold перед подсчётом
    preds = (torch.sigmoid(logits) > threshold).float()
    intersection = (preds * labels).sum()
    return (2.0 * intersection + eps) / (preds.sum() + labels.sum() + eps)


# Демонстрация: для одной и той же пары (logits, labels) lloss != 1 - score
torch.manual_seed(SEED)
logits = torch.randn(1, 1, 16, 16)
labels = (torch.rand(1, 1, 16, 16) > 0.5).float()

loss_val = dice_loss(logits, labels).item()
score_val = dice_score_metric(logits, labels).item()

print(f'dice_loss  : {loss_val:.4f}  (использует soft sigmoid, eps=1.0)')
print(f'dice_score : {score_val:.4f}  (использует threshold=0.5)')
print(f'1 - score  : {1 - score_val:.4f}  (НЕ равно dice_loss!)')
print()
print('Почему? loss — soft (вероятности), score — hard (бинарные).')
print('Loss дифференцируем (для backprop), score — нет.')

**Попробуй сам**: создай тензор logits = -1000 (модель предсказывает «фон везде») и labels с одной активной точкой. Посчитай dice_loss и dice_score — увидишь, что loss близок к 1.0 (плохо) и score = 0 (тоже плохо), но loss даёт градиент, а score — нет.

## 4. Focal Loss — для дисбаланса классов

[Focal Loss](https://arxiv.org/abs/1708.02002) — модификация BCE для борьбы с дисбалансом. Идея: уменьшить вклад «лёгких» примеров (которые модель уже уверенно классифицирует) и сосредоточить обучение на трудных.

$$\mathcal{L}_{FL}(y, \hat y) = -(1 - p_t)^\gamma \log(p_t), \qquad p_t = \begin{cases} \sigma(\hat y), & y = 1 \\ 1 - \sigma(\hat y), & y = 0 \end{cases}$$

При $\gamma = 0$ Focal Loss = BCE. При $\gamma > 0$ модель «фокусируется» на сложных пикселях. Обычно $\gamma = 2$.

Покажем на синтетическом примере: маска с 5% foreground и 95% background. Сравним BCE и Focal.

In [ ]:
def focal_loss(logits, labels, gamma=2.0, eps=1e-8):
    # Focal Loss с параметром gamma. Сумма по всем элементам
    p = torch.sigmoid(logits)
    p_t = torch.where(labels == 1, p, 1 - p)
    ce = -torch.log(p_t + eps)
    focal_weight = (1 - p_t) ** gamma
    return (focal_weight * ce).sum()


# Создадим сильно дисбалансированный пример: 5% маска
torch.manual_seed(SEED)
H = W = 64
labels = torch.zeros(1, 1, H, W)
# 5% foreground в правом нижнем углу
labels[..., -16:, -16:] = 1.0

# Сценарий: модель «лениво» предсказывает фон (логиты -2 везде)
logits_lazy = -2.0 * torch.ones_like(labels)

# Сценарий: модель отлично сегментирует foreground
logits_good = torch.where(labels == 1, torch.tensor(2.0), torch.tensor(-2.0))

bce_lazy = nn.BCEWithLogitsLoss(reduction='sum')(logits_lazy, labels).item()
bce_good = nn.BCEWithLogitsLoss(reduction='sum')(logits_good, labels).item()

fl_lazy = focal_loss(logits_lazy, labels, gamma=2).item()
fl_good = focal_loss(logits_good, labels, gamma=2).item()

print('Сценарий 1: модель ленивая (предсказывает фон везде, логит -2)')
print(f'   BCE:   {bce_lazy:.2f}')
print(f'   Focal: {fl_lazy:.2f}  (gamma=2)')
print()
print('Сценарий 2: модель отлично сегментирует')
print(f'   BCE:   {bce_good:.2f}')
print(f'   Focal: {fl_good:.2f}  (gamma=2)')
print()
print(f'Отношение Focal/BCE для ленивой модели: {fl_lazy / bce_lazy:.4f}')
print(f'Отношение Focal/BCE для хорошей модели: {fl_good / bce_good:.4f}')
print()
print('Видно, что Focal сильнее наказывает «ленивую» модель: лёгкие правильно')
print('предсказанные пиксели фона почти не вносят вклад, а ошибочно пропущенный')
print('foreground — да. У BCE вклад фона и foreground пропорционален количеству.')

**Попробуй сам**: меняй параметр `gamma` от 0 до 5. При gamma=0 получишь BCE. При gamma=5 — очень сильную фокусировку на сложных. На практике gamma=2 — хороший компромисс.

## 5. SegNet — упрощённая версия

[SegNet](https://arxiv.org/abs/1511.00561) — encoder-decoder сеть. Главная идея: **MaxPool возвращает индексы максимумов, симметричный MaxUnpool раскладывает признаки строго на эти позиции**. Это даёт sparse upsampling без обучаемых весов.

В оригинале — 5 уровней энкодера/декодера, depth 2-3. Здесь сделаем упрощённую версию: 4 уровня, depth=2, channels 3 -> 32 -> 64 -> 128 -> 256.

In [ ]:
class EncoderBlockMini(nn.Module):
    # Уменьшенный блок: depth=2 свёртки + MaxPool с индексами
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.maxpool = nn.MaxPool2d(2, stride=2, return_indices=True)

    def forward(self, x):
        x = self.conv(x)
        size_before = x.shape  # запоминаем форму до pooling — нужно в декодере
        x, indices = self.maxpool(x)
        return x, indices, size_before


class DecoderBlockMini(nn.Module):
    # Зеркальный блок: MaxUnpool + 2 свёртки. last=True убирает BN/ReLU из финального слоя
    def __init__(self, in_ch, out_ch, last=False):
        super().__init__()
        self.maxunpool = nn.MaxUnpool2d(2, stride=2)
        layers = [
            nn.Conv2d(in_ch, in_ch, 3, padding=1),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        ]
        if not last:
            layers += [nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        self.conv = nn.Sequential(*layers)

    def forward(self, x, indices, output_size):
        x = self.maxunpool(x, indices, output_size=output_size)
        return self.conv(x)


class MiniSegNet(nn.Module):
    # SegNet с 4 уровнями вместо 5. Channels: 3 -> 32 -> 64 -> 128 -> 256
    def __init__(self):
        super().__init__()
        self.enc0 = EncoderBlockMini(3, 32)
        self.enc1 = EncoderBlockMini(32, 64)
        self.enc2 = EncoderBlockMini(64, 128)
        self.enc3 = EncoderBlockMini(128, 256)

        self.dec3 = DecoderBlockMini(256, 128)
        self.dec2 = DecoderBlockMini(128, 64)
        self.dec1 = DecoderBlockMini(64, 32)
        self.dec0 = DecoderBlockMini(32, 1, last=True)

    def forward(self, x):
        x, i0, s0 = self.enc0(x)
        x, i1, s1 = self.enc1(x)
        x, i2, s2 = self.enc2(x)
        x, i3, s3 = self.enc3(x)

        x = self.dec3(x, i3, output_size=s3)
        x = self.dec2(x, i2, output_size=s2)
        x = self.dec1(x, i1, output_size=s1)
        x = self.dec0(x, i0, output_size=s0)
        return x  # логиты, БЕЗ sigmoid


# Smoke-test
torch.manual_seed(SEED)
seg_mini = MiniSegNet()
with torch.no_grad():
    out = seg_mini(torch.zeros(1, 3, 64, 64))
print(f'MiniSegNet: input (1,3,64,64) -> output {tuple(out.shape)}')
print(f'параметров: {sum(p.numel() for p in seg_mini.parameters()):,}')

assert out.shape == (1, 1, 64, 64)
print('ASSERT OK: MiniSegNet корректно сохраняет (H, W).')

## 6. UNet — упрощённая версия

[UNet](https://arxiv.org/abs/1505.04597) — стандарт de facto для медицинской сегментации. Главная идея: **skip-connections через конкатенацию** между симметричными уровнями энкодера и декодера.

В оригинале — 5 уровней с DoubleConv. Здесь — 3 уровня, channels 3 -> 32 -> 64 -> 128 (bottleneck).

In [ ]:
class DoubleConv(nn.Module):
    # Два Conv 3x3 с ReLU — стандартный блок UNet
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class MiniUNet(nn.Module):
    # UNet с 3 уровнями вместо 5. Channels: 3 -> 32 -> 64 -> 128 (bottleneck)
    def __init__(self, n_class=1):
        super().__init__()
        # Encoder
        self.conv1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        # Bottleneck
        self.conv3 = DoubleConv(64, 128)

        # Decoder с skip connections
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2_up = DoubleConv(128, 64)  # 64 (skip) + 64 (up) = 128 -> 64

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv1_up = DoubleConv(64, 32)   # 32 (skip) + 32 (up) = 64 -> 32

        self.final = nn.Conv2d(32, n_class, 1)

    def forward(self, x):
        # Encoder, сохраняем skip-features
        x1 = self.conv1(x);  x = self.pool1(x1)
        x2 = self.conv2(x);  x = self.pool2(x2)

        # Bottleneck
        x = self.conv3(x)

        # Decoder + concat
        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2_up(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv1_up(x)

        return self.final(x)  # логиты


# Smoke-test
torch.manual_seed(SEED)
unet_mini = MiniUNet()
with torch.no_grad():
    out = unet_mini(torch.zeros(1, 3, 64, 64))
print(f'MiniUNet: input (1,3,64,64) -> output {tuple(out.shape)}')
print(f'параметров: {sum(p.numel() for p in unet_mini.parameters()):,}')

assert out.shape == (1, 1, 64, 64)
print('ASSERT OK: MiniUNet корректно сохраняет (H, W).')

**Сравни параметры**: MiniSegNet и MiniUNet примерно одинакового размера. В полных версиях (5 уровней, depth=2-3) — около 29M (SegNet) и 31M (UNet) параметров. UNet немного больше из-за skip-connections (decoder получает на вход double channels).

**Попробуй сам**: добавь третий уровень в MiniUNet (4 уровня всего). Какие изменения нужны в декодере?

## 7. Демо обучения (опционально, ~30 секунд на CPU)

Сгенерируем синтетический датасет: изображения 64x64 с одним кругом на чёрном фоне, маска — этот круг. Обучим MiniUNet на 5 эпох с BCEWithLogitsLoss и убедимся, что лосс падает.

In [ ]:
def generate_synthetic_batch(batch_size=8, size=64, seed=None):
    # Генерирует батч: круги случайного размера/позиции на чёрном фоне
    if seed is not None:
        torch.manual_seed(seed)
    images = torch.zeros(batch_size, 3, size, size)
    masks = torch.zeros(batch_size, 1, size, size)
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing='ij')

    for b in range(batch_size):
        cy = torch.randint(15, size - 15, (1,)).item()
        cx = torch.randint(15, size - 15, (1,)).item()
        r = torch.randint(8, 14, (1,)).item()
        circle = ((yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2).float()
        # Изображение: серый фон + светлый круг
        images[b, 0] = 0.3 + 0.5 * circle
        images[b, 1] = 0.3 + 0.5 * circle
        images[b, 2] = 0.3 + 0.5 * circle
        # Шум
        images[b] += 0.05 * torch.randn(3, size, size)
        masks[b, 0] = circle
    return images.clamp(0, 1), masks


# Обучение
torch.manual_seed(SEED)
model = MiniUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

losses = []
ious = []
N_BATCHES_PER_EPOCH = 10

for epoch in range(1, 6):
    epoch_loss = 0.0
    epoch_iou = 0.0
    model.train()
    for batch in range(N_BATCHES_PER_EPOCH):
        x, y = generate_synthetic_batch(batch_size=8, seed=epoch * 100 + batch)
        x = x.to(device); y = y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
        # IoU после применения sigmoid + threshold
        with torch.no_grad():
            pred = (torch.sigmoid(logits) > 0.5).float()
            epoch_iou += iou_score_simple(pred.cpu(), y.cpu())

    avg_loss = epoch_loss / N_BATCHES_PER_EPOCH
    avg_iou = epoch_iou / N_BATCHES_PER_EPOCH
    losses.append(avg_loss)
    ious.append(avg_iou)
    print(f'Epoch {epoch}: BCE = {avg_loss:.4f}, IoU = {avg_iou:.4f}')

# Визуализация результата на новом батче
model.eval()
with torch.no_grad():
    x_demo, y_demo = generate_synthetic_batch(batch_size=4, seed=999)
    x_demo = x_demo.to(device)
    pred_demo = torch.sigmoid(model(x_demo)).cpu().numpy()
    x_demo = x_demo.cpu().numpy()
    y_demo = y_demo.cpu().numpy()

fig, axes = plt.subplots(2, 4 + 1, figsize=(15, 6))
axes[0, 0].plot(losses, marker='o', label='BCE loss');
axes[0, 0].set_title('Train BCE Loss'); axes[0, 0].set_xlabel('epoch'); axes[0, 0].grid()
axes[1, 0].plot(ious, marker='o', color='green', label='IoU');
axes[1, 0].set_title('Train IoU'); axes[1, 0].set_xlabel('epoch'); axes[1, 0].grid()

for i in range(4):
    axes[0, i + 1].imshow(x_demo[i].transpose(1, 2, 0))
    axes[0, i + 1].set_title(f'image {i}')
    axes[0, i + 1].axis('off')
    axes[1, i + 1].imshow(pred_demo[i, 0] > 0.5, cmap='gray')
    iou_demo = iou_score_simple(torch.tensor(pred_demo[i, 0] > 0.5), torch.tensor(y_demo[i, 0]))
    axes[1, i + 1].set_title(f'pred (IoU={iou_demo:.2f})')
    axes[1, i + 1].axis('off')

plt.tight_layout()
plt.show()

**Что должно произойти**:

- BCE loss падает с эпохами (от ~0.4 до <0.1).
- IoU растёт (от ~0.1 до 0.8+).
- На визуализации предсказанные маски всё точнее повторяют круги.

Если лосс не падает — проверь, что lr=1e-3 не слишком велик и что `model.train()` стоит перед `opt.step()`.

**Попробуй сам**: замени BCE на `dice_loss` или `focal_loss` (определены выше). Какой лосс быстрее сходится на этой простой задаче?

## 8. Бонусные лоссы — Tversky, Combo, Boundary

### Tversky Loss

[Tversky Loss](https://arxiv.org/abs/1706.05721) — обобщение Dice с двумя параметрами $\alpha$ и $\beta$, контролирующими штраф за FP и FN:

$$\mathcal{L}_{Tversky} = 1 - \frac{TP + \varepsilon}{TP + \alpha \cdot FP + \beta \cdot FN + \varepsilon}$$

При $\alpha = \beta = 0.5$ получаем Dice. Делая $\beta > \alpha$ (например, 0.7/0.3), штрафуем FN сильнее — модель меньше пропускает foreground (полезно в медицине).

### Combo Loss

Линейная комбинация BCE и Dice — берёт лучшее от обоих:

$$\mathcal{L}_{Combo} = \lambda \cdot \mathcal{L}_{BCE} + (1 - \lambda) \cdot \mathcal{L}_{Dice}$$

### Boundary Loss

[Boundary Loss](https://arxiv.org/abs/1812.07032) — фокусируется на границах через distance transform. Идея: умножаем (1 - prediction) на signed distance map и суммируем — модель тем сильнее штрафуется, чем дальше пиксель от истинной границы.

In [ ]:
def tversky_loss(logits, labels, alpha=0.3, beta=0.7, eps=1.0):
    # Tversky Loss. alpha штрафует FP, beta штрафует FN. При alpha=beta=0.5 = Dice
    p = torch.sigmoid(logits)
    tp = (p * labels).sum()
    fp = (p * (1 - labels)).sum()
    fn = ((1 - p) * labels).sum()
    return 1.0 - (tp + eps) / (tp + alpha * fp + beta * fn + eps)


def combo_loss(logits, labels, lam=0.5):
    # Combo Loss: lambda*BCE + (1-lambda)*Dice
    bce = nn.BCEWithLogitsLoss(reduction='mean')(logits, labels)
    dice = dice_loss(logits, labels)
    return lam * bce + (1.0 - lam) * dice


def boundary_loss_simple(logits, labels):
    # Упрощённый boundary loss: использует distance_transform_edt из scipy
    from scipy.ndimage import distance_transform_edt
    p = torch.sigmoid(logits)
    # Считаем signed distance map для каждой маски в батче
    labels_np = labels.detach().cpu().numpy().astype(np.uint8)
    dist_maps = np.zeros_like(labels_np, dtype=np.float32)
    for b in range(labels_np.shape[0]):
        mask = labels_np[b, 0]
        if mask.any() and (1 - mask).any():
            pos = distance_transform_edt(mask)
            neg = distance_transform_edt(1 - mask)
            dist_maps[b, 0] = neg - pos  # > 0 снаружи, < 0 внутри
    dist_t = torch.from_numpy(dist_maps).to(logits.device)
    # Штраф пропорционален расстоянию до границы
    return (p * dist_t).mean()


# Демо: посчитаем все три лосса на одном тензоре
torch.manual_seed(SEED)
logits = torch.randn(2, 1, 32, 32)
labels = (torch.rand(2, 1, 32, 32) > 0.6).float()

print(f'Tversky (alpha=0.3, beta=0.7) : {tversky_loss(logits, labels, 0.3, 0.7).item():.4f}')
print(f'Tversky (alpha=0.5, beta=0.5) : {tversky_loss(logits, labels, 0.5, 0.5).item():.4f}  (= Dice)')
print(f'Dice                          : {dice_loss(logits, labels).item():.4f}')
print(f'Combo (lambda=0.5)            : {combo_loss(logits, labels, 0.5).item():.4f}')
print(f'Boundary                      : {boundary_loss_simple(logits, labels).item():.4f}')

# Проверим, что все лоссы возвращают конечные числа (не NaN, не inf)
for name, val in [
    ('Tversky', tversky_loss(logits, labels)),
    ('Combo',   combo_loss(logits, labels)),
    ('Boundary', boundary_loss_simple(logits, labels)),
]:
    assert torch.isfinite(val), f'{name} loss вернул NaN/inf!'
print()
print('ASSERT OK: все бонусные лоссы возвращают конечные значения.')

**Попробуй сам**:

1. Сравни Tversky с разными (alpha, beta): (0.3, 0.7), (0.5, 0.5), (0.7, 0.3). На какие ошибки реагирует каждый?
2. Замени BCE на Combo Loss в демо-обучении выше. Сравни кривые сходимости.
3. Реализуй Tversky-Focal: $\mathcal{L}_{TF} = (1 - T)^\gamma$, где $T$ — Tversky index. Эта модификация ещё сильнее фокусируется на трудных примерах.

## Заключение

В этом конспекте мы прошли всю цепочку семантической сегментации:

1. **Метрики** IoU и Dice — попиксельные альтернативы accuracy. Связаны через $\text{Dice} = 2 \cdot \text{IoU}/(1 + \text{IoU})$.
2. **Лоссы**:
   - BCE — стабильная при больших логитах, оптимизируем через `BCEWithLogitsLoss`.
   - Dice — прямая оптимизация целевой метрики, soft-версия для дифференцируемости.
   - Focal — фокус на сложных примерах при дисбалансе.
3. **Архитектуры**:
   - SegNet — pooling indices, дёшев по памяти.
   - UNet — skip-connections, лучше сохраняет геометрию, стандарт в медицине.
4. **Бонусные лоссы** для специальных случаев: Tversky (контроль FP/FN), Combo (BCE+Dice), Boundary (фокус на границах).

**Ключевые принципы PyTorch для сегментации:**

- Всегда работаем с **логитами**, сигмоиду применяем только перед метрикой.
- `BCEWithLogitsLoss` стабильнее ручной BCE.
- Лучшие веса сохраняем через `copy.deepcopy(state_dict())` — простое присваивание не работает (модель потом меняет state).
- Между эпохами `iou_metric.reset()` — иначе статистика накапливается.

**Полный пайплайн в `solution.ipynb`** обучает 7 моделей: SegNet и UNet на трёх лоссах + бонусный Tversky. Лучшая комбинация — **UNet + BCE**, test IoU = 0.8481 на датасете PH2 (200 дерматоскопических снимков 256x256). Обучение занимает ~5 минут на MPS / GPU.